# Worked example: monitoring a small-business lending model

This notebook runs every check in the guide on **synthetic data** sized like a small community lender
(about 60 applications a month, a development sample of 1,200 booked loans). Nothing here comes from a real
lender, applicant, or model.

The simulated portfolio contains three problems for the checks to find:

1. A **data-feed problem**: after a system update, `debt_service_coverage` arrives blank on some files.
2. An **applicant-mix shift**: a new referral partner sends many more young businesses.
3. A **proxy candidate**: `zip_median_income_k` tracks estimated group membership much more strongly than it predicts repayment.

It also contains a cohort booked six months ago that is defaulting early at a higher rate than the model expects.

Sections follow the guide:

| Notebook section | Guide chapter | Cadence |
|---|---|---|
| 1. Data quality | 02 | Monthly |
| 2. Stability (PSI / CSI) | 02 | Monthly or quarterly |
| 3. Decision rates | 03 | Monthly |
| 4. Early warning | 03 | Quarterly |
| 5. Performance and calibration | 04 | Annually, or once a cohort matures |
| 6. Fair-lending review | 05 | Quarterly (outcomes), annually (inputs) |
| 7. Summary for the monitoring report | 06, 08 | Each cycle |

In [1]:
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from synthetic_data import generate, INPUTS
from monitoring_checks import (
    make_bins, bin_shares, psi, psi_noise_threshold, stability_report, data_quality_report,
    decision_rates, early_warning_ratio, auc_with_ci, ks_statistic, calibration_table,
    adverse_impact_ratio, proxy_screen, lda_search,
)

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)
data = generate(seed=7)
{k: len(v) for k, v in data.items()}

{'development': 1200,
 'q1': 180,
 'q2': 190,
 'matured_cohort': 420,
 'seasoning_cohort': 360}

## 0. The model being monitored

A simple logistic scorecard fit on the development sample. In practice this could be your own model,
a vendor score, or a scorecard in your loan-origination system; the checks only need the inputs, the score,
the decision, and eventually the outcome.

Applicants are approved when their predicted default probability (PD) is below the cut-off that approved
70% of the development sample.

In [2]:
dev = data["development"]
model = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(max_iter=1000))
model.fit(dev[INPUTS], dev["default_12m"])

cutoff = np.quantile(model.predict_proba(dev[INPUTS])[:, 1], 0.70)
rng = np.random.default_rng(1)
for name, df in data.items():
    df["pd"] = model.predict_proba(df[INPUTS])[:, 1]
    df["approved"] = (df["pd"] <= cutoff).astype(int)
    df["override"] = rng.binomial(1, 0.06, len(df))   # simulated manual overrides

print(f"Development default rate: {dev['default_12m'].mean():.1%}   PD cut-off: {cutoff:.3f}")

Development default rate: 19.2%   PD cut-off: 0.227


## 1. Data quality (monthly)

The cheapest check and often the most useful. A blank or out-of-range field silently changes every score it
touches, and nothing downstream will tell you.

In [3]:
print("Quarter 1")
display(data_quality_report(dev, data["q1"], INPUTS))
print("Quarter 2")
display(data_quality_report(dev, data["q2"], INPUTS))

Quarter 1


,variable,missing_baseline,missing_current,share_outside_baseline_range,status
0,credit_score,0.0,0.0,0.0111,green
1,years_in_business,0.0,0.0,0.0000,green
2,log_annual_revenue,0.0,0.0,0.0000,green
3,debt_service_coverage,0.0,0.0,0.0000,green
4,prior_delinquencies,0.0,0.0,0.0000,green
5,zip_median_income_k,0.0,0.0,0.0000,green


Quarter 2


,variable,missing_baseline,missing_current,share_outside_baseline_range,status
0,credit_score,0.0,0.0000,0.0053,green
1,years_in_business,0.0,0.0000,0.0000,green
2,log_annual_revenue,0.0,0.0000,0.0053,green
3,debt_service_coverage,0.0,0.1579,0.0000,red
4,prior_delinquencies,0.0,0.0000,0.0000,green
5,zip_median_income_k,0.0,0.0000,0.0000,green


**Reading it:** in Quarter 2, `debt_service_coverage` went from 0% to about 16% missing. That is a **red** data
alert. The owner of the data feed investigates first. Here the cause is a system update. The model keeps
scoring those files by filling in the median, so those applicants are being scored as if they had typical coverage.

## 2. Stability: PSI and CSI

**Population Stability Index (PSI)** compares today's distribution of the score (or an input, where it is called
CSI) against the development sample.

The common rule of thumb (below 0.10 stable, 0.10–0.25 watch, above 0.25 act) was developed for large
portfolios. **At small volumes, sampling noise alone produces PSI values above those thresholds.**
The simulation below draws samples with *no* real shift and records the PSI.

In [4]:
sim_rng = np.random.default_rng(0)
base = pd.Series(sim_rng.normal(size=1200))
rows = []
for n_current in [60, 180, 1200]:
    for n_bins in [5, 10]:
        edges = make_bins(base, n_bins)
        exp = bin_shares(base, edges, include_missing=False)
        vals = [psi(exp, bin_shares(pd.Series(sim_rng.normal(size=n_current)), edges, include_missing=False))
                for _ in range(2000)]
        rows.append({"applications in period": n_current, "bins": n_bins,
                     "median PSI (no real shift)": np.median(vals),
                     "99th pct PSI (no real shift)": np.quantile(vals, 0.99),
                     "formula threshold (99%)": psi_noise_threshold(1200, n_current, n_bins)})
pd.DataFrame(rows).round(3)

,applications in period,bins,median PSI (no real shift),99th pct PSI (no real shift),formula threshold (99%)
0,60,5,0.061,0.250,0.232
1,60,10,0.155,0.817,0.379
2,180,5,0.022,0.086,0.085
3,180,10,0.052,0.130,0.138
4,1200,5,0.006,0.020,0.022
5,1200,10,0.011,0.027,0.036


**Reading it:**

* With one month of applications (about 60) and 10 bins, the *median* PSI is about 0.17 even though nothing
  changed. Under the rule of thumb you would be investigating most months for no reason.
* Using **5 bins** and **pooling a quarter** (about 180 applications) brings the noise floor down to roughly 0.09.
* `psi_noise_threshold` gives a quick approximation of that noise floor. The guide's rule is to alert at
  `max(rule of thumb, noise threshold)`. The approximation understates the noise when bins hold only a
  handful of applicants (60 applicants in 10 bins), which is another reason to use fewer bins.

In [5]:
cols = ["pd"] + INPUTS
print("Quarter 1 vs development")
display(stability_report(dev, data["q1"], cols))
print("Quarter 2 vs development")
display(stability_report(dev, data["q2"], cols))

Quarter 1 vs development


,variable,psi,noise_threshold_99,amber_at,red_at,missing_rate_baseline,missing_rate_current,status
0,pd,0.0680,0.0964,0.1,0.25,0.0,0.0,green
1,credit_score,0.0471,0.0964,0.1,0.25,0.0,0.0,green
2,years_in_business,0.0656,0.0964,0.1,0.25,0.0,0.0,green
3,log_annual_revenue,0.0152,0.0964,0.1,0.25,0.0,0.0,green
4,debt_service_coverage,0.0250,0.0964,0.1,0.25,0.0,0.0,green
5,prior_delinquencies,0.0047,0.0725,0.1,0.25,0.0,0.0,green
6,zip_median_income_k,0.0277,0.0964,0.1,0.25,0.0,0.0,green


Quarter 2 vs development


,variable,psi,noise_threshold_99,amber_at,red_at,missing_rate_baseline,missing_rate_current,status
0,pd,0.0390,0.0920,0.1,0.25,0.0,0.0000,green
1,credit_score,0.0128,0.0920,0.1,0.25,0.0,0.0000,green
2,years_in_business,0.2145,0.0920,0.1,0.25,0.0,0.0000,amber
3,log_annual_revenue,0.0059,0.0920,0.1,0.25,0.0,0.0000,green
4,debt_service_coverage,1.2197,0.0920,0.1,0.25,0.0,0.1579,red
5,prior_delinquencies,0.0013,0.0692,0.1,0.25,0.0,0.0000,green
6,zip_median_income_k,0.0195,0.0920,0.1,0.25,0.0,0.0000,green


**Reading it:**

* Quarter 1 is green throughout.
* Quarter 2: `debt_service_coverage` is red (the missing values found in section 1). `years_in_business` is
  **amber**, which is the new referral channel bringing in younger businesses.
* The **score** PSI stays green even though an input moved. That is not reassuring on its own. It can mean the
  shift in one input is being offset by another, or, as here, that blank values replaced with the median are
  pulling scores toward the middle. The action is to check whether the model was ever fitted on businesses
  this young (see guide chapter 02, "score stable, inputs moving").

## 3. Decision rates (monthly)

Approval and override rates move before anything else does. They are also what your board and funders
already watch.

In [6]:
pd.DataFrame({"Quarter 1": decision_rates(data["q1"]), "Quarter 2": decision_rates(data["q2"])}).T

,applications,approval_rate,override_rate
Quarter 1,180.0,0.6000,0.0500
Quarter 2,190.0,0.6474,0.0579


**Reading it:** the approval rate rose from 60% to about 65% in Quarter 2. With `debt_service_coverage` blank on
16% of files and replaced by the median, weaker applicants look average and get through. This is the
data alert from section 1 showing up in lending decisions. Fixing the feed comes before any change to the model.

## 4. Early warning before outcomes mature (quarterly)

Small-business defaults can take 12 months or more to show up. Waiting for them means a problem can run for a
year. The early-warning check compares **early delinquency** (30+ days past due at 6 months on book) with the
number the model implies.

`early_to_final_ratio` is the share of 12-month defaults that were already 30+ days past due at 6 months, measured
on the development sample (or your own history).

In [7]:
early_to_final = dev["dpd30_at_6mob"].sum() / dev["default_12m"].sum()
print(f"Share of eventual defaults already 30+ DPD at 6 months: {early_to_final:.0%}")

pd.DataFrame({
    "Cohort booked 12 months ago": early_warning_ratio(data["matured_cohort"], "pd", "dpd30_at_6mob", early_to_final),
    "Cohort booked 6 months ago": early_warning_ratio(data["seasoning_cohort"], "pd", "dpd30_at_6mob", early_to_final),
}).T

Share of eventual defaults already 30+ DPD at 6 months: 41%


,loans,expected,actual,ratio,p_value,status
Cohort booked 12 months ago,420,33.0,27,0.82,0.8747,green
Cohort booked 6 months ago,360,28.6,49,1.71,0.0003,red


**Reading it:** the older cohort ran at about 0.8× expected (green). The cohort booked six months ago is at about
**1.7× expected**, and the Poisson test says that is very unlikely to be chance. That is a red early warning,
six months before the 12-month default rate would show it. The next step is to work out *which* loans are
driving it: channel, industry, geography, or a policy change at the time they were booked.

The check returns `insufficient` rather than a ratio when fewer than 5 early defaults are expected. With so few
expected, one or two loans would swing the ratio on their own.

## 5. Performance and calibration (once a cohort matures)

Two questions:

* **Does the model still rank risk well?** Measured by AUC and KS, *with confidence intervals*.
* **Are its default probabilities the right level?** This is calibration: expected vs actual defaults by band.

In [8]:
mc = data["matured_cohort"]
perf = pd.DataFrame({
    "Development": auc_with_ci(dev["default_12m"], dev["pd"]),
    "Matured cohort": auc_with_ci(mc["default_12m"], mc["pd"]),
}).T
perf["ks"] = [ks_statistic(dev["default_12m"], dev["pd"]), ks_statistic(mc["default_12m"], mc["pd"])]
perf.round(3)

,defaults,non_defaults,auc,ci_low,ci_high,ks
Development,230.0,970.0,0.722,0.682,0.762,0.330
Matured cohort,63.0,357.0,0.755,0.683,0.828,0.404


**Reading it:** the matured cohort has only about 60 defaults, so its AUC confidence interval spans roughly
0.68–0.83, a range of about 0.15. Here the point estimate is actually *above* development, but a
drop of a few hundredths would sit just as comfortably inside that interval. With this few defaults,
**don't act on a point drop alone**. The guide's rule is to go amber only when the drop
is larger than 0.03 *and* development AUC lies outside the cohort's interval.

In [9]:
calibration_table(mc, "pd", "default_12m", n_bands=5)

,pd_band,loans,expected,actual,actual_to_expected,ratio_ci_low,ratio_ci_high
0,1.1% - 8.3%,84,4.6,1,0.22,0.01,1.22
1,8.3% - 12.5%,84,8.7,7,0.80,0.32,1.65
2,12.5% - 18.5%,84,13.1,9,0.69,0.31,1.31
3,18.5% - 27.8%,84,19.5,18,0.92,0.55,1.46
4,27.8% - 81.6%,84,34.1,28,0.82,0.55,1.19


In [10]:
from scipy import stats
expected, actual = mc["pd"].sum(), mc["default_12m"].sum()
lo = stats.chi2.ppf(0.025, 2 * actual) / 2 / expected
hi = stats.chi2.ppf(0.975, 2 * (actual + 1)) / 2 / expected
print(f"Overall: {actual} actual vs {expected:.1f} expected defaults -> ratio {actual/expected:.2f} (95% CI {lo:.2f}-{hi:.2f})")

Overall: 63 actual vs 80.0 expected defaults -> ratio 0.79 (95% CI 0.61-1.01)


**Reading it:** the actual-to-expected ratio moves around between bands, but every band's 95% interval includes
1.0. Overall (next cell), the cohort had fewer defaults than the model predicted, a ratio of about 0.8, but that
interval also includes 1.0. That makes it a **watch item**: if the next matured cohort also comes in low, the model is
over-predicting and should be recalibrated. Recalibrate when the ratio is consistently off in one direction while rank order holds;
rebuild when rank order itself degrades.

## 6. Fair-lending review

### 6a. Outcome check (quarterly)

Lenders that do not collect race or ethnicity (most non-mortgage lenders) commonly estimate it with
**BISG**, which combines surname and geography. Here `group_prob` is a *simulated* stand-in for that estimated
probability. The adverse impact ratio (AIR) weights each applicant by their probability rather than assigning
them to one group.

In [11]:
pd.DataFrame({"Development sample": adverse_impact_ratio(dev, "group_prob")}).T

,effective_n_group,effective_n_comparison,approval_rate_group,approval_rate_comparison,air,p_value,status
Development sample,537.3,662.7,0.6558,0.7358,0.891,0.0026,amber


**Reading it:** the group's approval rate is about 89% of the comparison group's (AIR ≈ 0.89), and the difference is
statistically significant. Under the chapter 06 thresholds that is **amber**. It is not a finding of discrimination.
It is a prompt to find out *which inputs* drive the gap, which the next step does.

### 6b. Input-level proxy review (annually, and whenever inputs change)

For each input: how strongly does it track estimated group membership, and how much does it actually tell you about
repayment?

In [12]:
screen = proxy_screen(dev, INPUTS, "group_prob", "default_12m")
screen

,input,corr_with_group_prob,univariate_auc,review
5,zip_median_income_k,-0.520,0.552,yes
0,credit_score,-0.086,0.635,no
4,prior_delinquencies,0.051,0.578,no
3,debt_service_coverage,0.033,0.578,no
2,log_annual_revenue,-0.030,0.522,no
1,years_in_business,0.024,0.588,no


**Reading it:** `zip_median_income_k` is correlated with estimated group membership (about −0.5) but has a
univariate AUC barely above 0.5. That means strong association with the group and weak predictive value, so it goes to
review. The review asks two questions: is there a documented business reason to use this input, and is there an
alternative that does the same job with less disparity?

### 6c. Less-discriminatory-alternative (LDA) search

Refit without each candidate and compare on hold-out data, repeated over 20 random splits so the
comparison isn't driven by one lucky or unlucky sample. Each variant approves the same share of applicants.

In [13]:
lda = lda_search(dev, INPUTS, candidates=["zip_median_income_k", "credit_score", "prior_delinquencies"],
                 outcome_col="default_12m", group_prob_col="group_prob", approve_share=0.70)
lda

,variant,auc,air,auc_change,auc_change_sd,air_change,air_change_sd
0,current model,0.7086,0.9005,0.0000,0.0000,0.0000,0.0000
1,without zip_median_income_k,0.7044,0.9453,-0.0043,0.0047,0.0448,0.0202
2,without credit_score,0.6529,0.9005,-0.0558,0.0161,0.0000,0.0271
3,without prior_delinquencies,0.6862,0.8822,-0.0224,0.0178,-0.0183,0.0299


**Reading it:**

* **Without `zip_median_income_k`**, AUC drops by less than 0.01, within split-to-split noise, while AIR improves
  by about 0.04. That is a candidate less-discriminatory alternative, and the review should document it and the
  decision taken.
* **Without `credit_score`**, AUC falls by about 0.06 and AIR does not improve. The input carries real predictive
  value and is not driving the gap.
* **Without `prior_delinquencies`**, AUC falls and AIR gets slightly *worse*.

Whatever is decided, the record should show what was tested, what the numbers were, and why the chosen model
was kept or changed (see `templates/fair-lending-review-template.md`).

## 7. Summary for the monitoring report

One row per check, status, and owner. This is the table that goes at the top of the monitoring report
(`templates/monitoring-report-template.md`).

In [14]:
dq = data_quality_report(dev, data["q2"], INPUTS)
st = stability_report(dev, data["q2"], ["pd"] + INPUTS)
ew = early_warning_ratio(data["seasoning_cohort"], "pd", "dpd30_at_6mob", early_to_final)
air = adverse_impact_ratio(dev, "group_prob")

summary = pd.DataFrame([
    {"check": "Data quality: debt_service_coverage missing", "result": f"{dq.set_index('variable').loc['debt_service_coverage','missing_current']:.0%} missing (baseline 0%)",
     "status": dq.set_index('variable').loc['debt_service_coverage','status'], "owner": "Data / LOS administrator", "action": "Investigate feed; fix before next cycle"},
    {"check": "Stability: years_in_business CSI", "result": f"{st.set_index('variable').loc['years_in_business','psi']:.2f}",
     "status": st.set_index('variable').loc['years_in_business','status'], "owner": "Model owner", "action": "Check model coverage of young businesses; monitor new channel"},
    {"check": "Stability: score PSI", "result": f"{st.set_index('variable').loc['pd','psi']:.2f}",
     "status": st.set_index('variable').loc['pd','status'], "owner": "Model owner", "action": "None beyond input follow-ups"},
    {"check": "Early warning: 6-month cohort", "result": f"{ew['ratio']}x expected early delinquency",
     "status": ew['status'], "owner": "Chief lending officer", "action": "Segment the cohort; report to credit committee"},
    {"check": "Fair lending: approval AIR", "result": f"{air['air']:.2f}",
     "status": air['status'], "owner": "Compliance / CLO", "action": "Input review done; document LDA decision on zip income"},
])
summary

,check,result,status,owner,action
0,Data quality: debt_service_coverage missing,16% missing (baseline 0%),red,Data / LOS administrator,Investigate feed; fix before next cycle
1,Stability: years_in_business CSI,0.21,amber,Model owner,Check model coverage of young businesses; moni...
2,Stability: score PSI,0.04,green,Model owner,None beyond input follow-ups
3,Early warning: 6-month cohort,1.71x expected early delinquency,red,Chief lending officer,Segment the cohort; report to credit committee
4,Fair lending: approval AIR,0.89,amber,Compliance / CLO,Input review done; document LDA decision on zi...
